# Tiền Xử Lý Dữ Liệu (Data Preprocessing)

## Các bước xử lý:
1. **Xóa các kí hiệu thừa trong công thức**
2. **Chuyển prep_time & cook_time** từ text → phút (numeric)
3. **Binning cột servings** thành categories theo quy mô công thức
4. **Xử lý missing values** trong cột views
5. **Encode cột difficulty** thành numeric (1-3)
6. **Impute cook_time_min** bằng cách tìm công thức tương tự hoặc sử dụng LLM
7. **Lưu kết quả** vào `recipes_processed.json` và tóm tắt kết quả vào `preprocessing_summary.json`

## Khai báo thư viện


In [1]:
import json
import os
import re
import statistics
import unicodedata
import warnings
from collections import Counter, defaultdict
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd
import requests

warnings.filterwarnings('ignore')


## Cấu hình chạy


In [ ]:
INPUT_PATH = Path('data/recipes_detail.json')
OUTPUT_PATH = Path('data/recipes_processed.json')
OUTPUT_2_PATH = Path('dataset/recipes_processed.json')
SUMMARY_PATH = Path('data/recipes_processed_summary.json')
DOTENV_PATH = '.env'

PLACEHOLDER_VALUES = ['..', '...', '....']

# USE_LLM_FALLBACK = False
USE_LLM_FALLBACK = True
LLM_MODEL = 'gpt-4.1-mini'
LLM_LIMIT = None
# LLM_LIMIT = 30
LLM_TIMEOUT_SECONDS = 60


## Xóa placeholder bất thường trong nguyên liệu


In [3]:
# Load các file JSON
with open('unique_ingredients.json', 'r', encoding='utf-8') as f:
    unique_ingredients = json.load(f)

with open('ingredients_synonyms_qwen.json', 'r', encoding='utf-8') as f:
    ingredients_synonyms = json.load(f)

with open('recipes_detail.json', 'r', encoding='utf-8') as f:
    recipes_detail = json.load(f)

print("=== Trước khi xóa ===")
print(f"unique_ingredients.json: {len(unique_ingredients)} items")
print(f"ingredients_synonyms_qwen.json: {len(ingredients_synonyms)} items")
print(f"recipes_detail.json: {len(recipes_detail)} recipes")


=== Trước khi xóa ===
unique_ingredients.json: 7947 items
ingredients_synonyms_qwen.json: 7947 items
recipes_detail.json: 13624 recipes


In [4]:
# 1. Xóa từ unique_ingredients.json
unique_ingredients_clean = [item for item in unique_ingredients if item not in PLACEHOLDER_VALUES]

# 2. Xóa từ ingredients_synonyms_qwen.json
ingredients_synonyms_clean = [
    item for item in ingredients_synonyms
    if item.get('ingredient') not in PLACEHOLDER_VALUES
]

# 3. Xóa từ recipes_detail.json - loại bỏ placeholder ingredients
for recipe in recipes_detail:
    if 'ingredients' in recipe and isinstance(recipe['ingredients'], list):
        recipe['ingredients'] = [
            ing for ing in recipe['ingredients']
            if isinstance(ing, dict) and ing.get('name') not in PLACEHOLDER_VALUES
        ]

print("=== Sau khi xóa ===")
print(f"unique_ingredients.json: {len(unique_ingredients_clean)} items")
print(f"ingredients_synonyms_qwen.json: {len(ingredients_synonyms_clean)} items")
print(f"recipes_detail.json: {len(recipes_detail)} recipes")


=== Sau khi xóa ===
unique_ingredients.json: 7947 items
ingredients_synonyms_qwen.json: 7947 items
recipes_detail.json: 13624 recipes


In [5]:
# Lưu các file đã làm sạch
with open('unique_ingredients.json', 'w', encoding='utf-8') as f:
    json.dump(unique_ingredients_clean, f, ensure_ascii=False, indent=2)

with open('ingredients_synonyms_qwen.json', 'w', encoding='utf-8') as f:
    json.dump(ingredients_synonyms_clean, f, ensure_ascii=False, indent=2)

with open('recipes_detail.json', 'w', encoding='utf-8') as f:
    json.dump(recipes_detail, f, ensure_ascii=False, indent=2)


## Load dữ liệu


In [6]:
with open(INPUT_PATH, 'r', encoding='utf-8') as f:
    recipes_data = json.load(f)

df = pd.DataFrame(recipes_data)

print('THÔNG TIN DỮ LIỆU TRƯỚC XỬ LÝ')
print(f"Kích thước: {df.shape[0]} hàng × {df.shape[1]} cột")
print(sorted(df.columns.tolist()))
print('Một vài dòng dữ liệu mẫu:')
print(df[['dish_name', 'prep_time', 'cook_time', 'servings', 'difficulty', 'views']].head(3))


THÔNG TIN DỮ LIỆU TRƯỚC XỬ LÝ
Kích thước: 13624 hàng × 10 cột
['category', 'cook_time', 'difficulty', 'dish_name', 'ingredients', 'instructions', 'prep_time', 'servings', 'url', 'views']
Một vài dòng dữ liệu mẫu:
                dish_name prep_time      cook_time servings  difficulty  views
0          Vịt quay tỳ bà    10 giờ        25 phút      4-5         Khó   16.0
1  Dẻ sườn bò nướng sa tế   20 phút  1 giờ 20 phút      2-3          Dễ    1.0
2   thịt heo quay tại nhà   25 phút          1 giờ      800  Trung bình    1.0


## Xử lý prep_time và cook_time


In [7]:
def convert_time_to_minutes(time_str):
    """
    Chuyển chuỗi thời gian thành số phút
    Ví dụ:
    - "10 giờ" -> 600
    - "25 phút" -> 25
    - "1 giờ 20 phút" -> 80
    """
    if pd.isna(time_str) or time_str is None:
        return 0

    time_str = str(time_str).strip()
    minutes = 0

    hours_match = re.search(r'(\d+)\s*giờ', time_str)
    if hours_match:
        minutes += int(hours_match.group(1)) * 60

    minutes_match = re.search(r'(\d+)\s*phút', time_str)
    if minutes_match:
        minutes += int(minutes_match.group(1))

    return minutes


df['prep_time_min'] = df['prep_time'].apply(convert_time_to_minutes)
df['cook_time_min'] = df['cook_time'].apply(convert_time_to_minutes)

print("BƯỚC 1: XỬ LÝ PREP_TIME & COOK_TIME")
print(f"\nThống kê prep_time_min (phút):")
print(f"  - Min: {df['prep_time_min'].min()}")
print(f"  - Max: {df['prep_time_min'].max()}")
print(f"  - Mean: {df['prep_time_min'].mean():.2f}")
print(f"  - Median: {df['prep_time_min'].median():.2f}")

print(f"\nThống kê cook_time_min (phút):")
print(f"  - Min: {df['cook_time_min'].min()}")
print(f"  - Max: {df['cook_time_min'].max()}")
print(f"  - Mean: {df['cook_time_min'].mean():.2f}")
print(f"  - Median: {df['cook_time_min'].median():.2f}")

print(f"\nMột vài ví dụ chuyển đổi:")
for i in range(5):
    print(f"  prep_time: '{df['prep_time'].iloc[i]}' -> {df['prep_time_min'].iloc[i]} ph?t")
    print(f"  cook_time: '{df['cook_time'].iloc[i]}' -> {df['cook_time_min'].iloc[i]} ph?t")


BƯỚC 1: XỬ LÝ PREP_TIME & COOK_TIME

Thống kê prep_time_min (phút):
  - Min: 1
  - Max: 1380
  - Mean: 33.68
  - Median: 20.00

Thống kê cook_time_min (phút):
  - Min: 0
  - Max: 1439
  - Mean: 31.47
  - Median: 20.00

Một vài ví dụ chuyển đổi:
  prep_time: '10 giờ' -> 600 ph?t
  cook_time: '25 phút' -> 25 ph?t
  prep_time: '20 phút' -> 20 ph?t
  cook_time: '1 giờ 20 phút' -> 80 ph?t
  prep_time: '25 phút' -> 25 ph?t
  cook_time: '1 giờ' -> 60 ph?t
  prep_time: '45 phút' -> 45 ph?t
  cook_time: '1 giờ' -> 60 ph?t
  prep_time: '2 phút' -> 2 ph?t
  cook_time: '8 phút' -> 8 ph?t


## Binning cột servings theo logic


In [8]:
def normalize_servings(servings_str):
    if pd.isna(servings_str) or servings_str is None:
        return pd.NA

    servings_str = str(servings_str).strip()
    if not servings_str:
        return pd.NA

    servings_str = servings_str.lstrip('0') or '0'
    if servings_str == '0':
        return pd.NA

    return servings_str


def extract_servings_numeric(servings_str):
    """
    Chuyển chuỗi servings thành số numerics
    Ví dụ:
    - "2-4" -> 3 (trung bình)
    - "4" -> 4
    - "02" -> 2
    """
    if pd.isna(servings_str) or servings_str is None:
        return 0

    servings_str = str(servings_str).strip()
    servings_str = servings_str.lstrip('0') or '0'

    if '-' in servings_str:
        parts = servings_str.split('-')
        try:
            start = int(parts[0].strip())
            end = int(parts[1].strip())
            return (start + end) / 2
        except:
            return 0

    try:
        return float(servings_str)
    except:
        return 0


def bin_servings(numeric_servings):
    if numeric_servings == 0:
        return 'Unknown'
    elif numeric_servings <= 1:
        return '1 người'
    elif numeric_servings < 3:
        return '2-3 người'
    elif numeric_servings < 5:
        return '4-6 người'
    elif numeric_servings <= 8:
        return '6-8 người'
    else:
        return 'Uncertain'


print("BƯỚC 2: BINNING CỘT SERVINGS")

invalid_zero_count = (df['servings'].astype('string').str.strip() == '0').sum()
df['servings'] = df['servings'].apply(normalize_servings)
missing_servings_count = df['servings'].isna().sum()
servings_mode = df['servings'].mode().iloc[0]
df['servings'] = df['servings'].fillna(servings_mode)

df['servings_numeric'] = df['servings'].apply(extract_servings_numeric)
df['servings_bin'] = df['servings_numeric'].apply(bin_servings)

print(f"\nSố giá trị '0' được xem là missing: {invalid_zero_count}")
print(f"Số missing sau khi chuẩn hóa: {missing_servings_count}")
print(f"Giá trị mode dùng để fill: {servings_mode}")

print(f"Thống kê servings_numeric (trước binning):")
print(f"  - Min: {df['servings_numeric'].min()}")
print(f"  - Max: {df['servings_numeric'].max()}")
print(f"  - Mean: {df['servings_numeric'].mean():.2f}")
print(f"  - Unique values: {df['servings_numeric'].nunique()}")

print(f"\nPhân bố sau binning:")
servings_dist = df['servings_bin'].value_counts().sort_index()
for category, count in servings_dist.items():
    percent = (count / len(df)) * 100
    print(f"  - {category}: {count} ({percent:.2f}%)")

print(f"\nMột vài ví dụ binning:")
for i in range(10):
    print(f"  '{df['servings'].iloc[i]}' -> numeric: {df['servings_numeric'].iloc[i]} -> bin: '{df['servings_bin'].iloc[i]}'")


BƯỚC 2: BINNING CỘT SERVINGS

Số giá trị '0' được xem là missing: 3
Số missing sau khi chuẩn hóa: 3
Giá trị mode dùng để fill: 4
Thống kê servings_numeric (trước binning):
  - Min: 1.0
  - Max: 970.0
  - Mean: 11.76
  - Unique values: 83

Phân bố sau binning:
  - 1 người: 1066 (7.82%)
  - 2-3 người: 3538 (25.97%)
  - 4-6 người: 7334 (53.83%)
  - 6-8 người: 1049 (7.70%)
  - Uncertain: 637 (4.68%)

Một vài ví dụ binning:
  '4-5' -> numeric: 4.5 -> bin: '4-6 người'
  '2-3' -> numeric: 2.5 -> bin: '2-3 người'
  '800' -> numeric: 800.0 -> bin: 'Uncertain'
  '1' -> numeric: 1.0 -> bin: '1 người'
  '1' -> numeric: 1.0 -> bin: '1 người'
  '1' -> numeric: 1.0 -> bin: '1 người'
  '500' -> numeric: 500.0 -> bin: 'Uncertain'
  '4' -> numeric: 4.0 -> bin: '4-6 người'
  '2-4' -> numeric: 3.0 -> bin: '4-6 người'
  '3-4' -> numeric: 3.5 -> bin: '4-6 người'


## Xử lý missing values


In [9]:
print("BƯỚC 3: XỬ LÝ MISSING VALUES - CỘT VIEWS")


print(f"\nTrước xử lý:")
print(f"  - Missing values: {df['views'].isnull().sum()}")
print(f"  - Tỷ lệ missing: {(df['views'].isnull().sum() / len(df) * 100):.2f}%")

df['views'] = df['views'].fillna(0).astype(int)

print(f"\nSau xử lý:")
print(f"  - Missing values: {df['views'].isnull().sum()}")
print(f"  - Số lượng views = 0: {(df['views'] == 0).sum()}")

print(f"\nThống kê views sau xử lý:")
print(f"  - Min: {df['views'].min()}")
print(f"  - Max: {df['views'].max()}")
print(f"  - Mean: {df['views'].mean():.2f}")
print(f"  - Median: {df['views'].median():.2f}")


BƯỚC 3: XỬ LÝ MISSING VALUES - CỘT VIEWS

Trước xử lý:
  - Missing values: 42
  - Tỷ lệ missing: 0.31%

Sau xử lý:
  - Missing values: 0
  - Số lượng views = 0: 42

Thống kê views sau xử lý:
  - Min: 0
  - Max: 1331941
  - Mean: 22410.67
  - Median: 6414.00


## Encoding cột difficulty


In [10]:
difficulty_mapping = {
    'Dễ': 1,
    'Trung bình': 2,
    'Khó': 3
}

df['difficulty'] = df['difficulty'].map(difficulty_mapping)

print("BƯỚC 4: ENCODE CỘT DIFFICULTY")

print(f"\nMapping:")
for text, numeric in difficulty_mapping.items():
    print(f"  - '{text}' -> {numeric}")

print(f"\nPhân bố difficulty:")
for numeric in sorted(df['difficulty'].dropna().unique()):
    count = (df['difficulty'] == numeric).sum()
    percent = (count / len(df)) * 100
    print(f"  - {numeric}: {count} ({percent:.2f}%)")


BƯỚC 4: ENCODE CỘT DIFFICULTY

Mapping:
  - 'Dễ' -> 1
  - 'Trung bình' -> 2
  - 'Khó' -> 3

Phân bố difficulty:
  - 1: 11908 (87.40%)
  - 2: 1678 (12.32%)
  - 3: 38 (0.28%)


## Chuẩn bị hàm cho bước impute cook_time_min


In [11]:
STOPWORDS = {
    'an', 'au', 'bac', 'banh', 'bot', 'ca', 'cach', 'canh', 'cay', 'chao',
    'chien', 'ga', 'gio', 'hau', 'heo', 'huong', 'kho', 'lam', 'lon', 'mon',
    'muoi', 'nam', 'ngon', 'nuoc', 'nuong', 'rau', 'sot', 'thit', 'tom', 'tron', 'xao'
}


def load_dotenv(dotenv_path):
    dotenv_path = Path(dotenv_path)
    if not dotenv_path.exists():
        return

    for raw_line in dotenv_path.read_text(encoding='utf-8', errors='ignore').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip().removeprefix('export ').strip()
        value = value.strip()
        if value and ((value[0] == value[-1]) and value[0] in {"'", '"'}):
            value = value[1:-1]
        os.environ.setdefault(key, value)


def normalize_text(text):
    text = unicodedata.normalize('NFKD', text or '')
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return ' '.join(text.split())


def tokenize(text, keep_stopwords=False):
    tokens = set(normalize_text(text).split())
    if keep_stopwords:
        return tokens
    return {token for token in tokens if len(token) > 1 and token not in STOPWORDS}


def ingredient_text(ingredients):
    names = []
    for item in ingredients or []:
        if isinstance(item, dict):
            name = item.get('name')
            if name:
                names.append(str(name))
        elif item:
            names.append(str(item))
    return ' '.join(names)


def flatten_instruction_text(instructions):
    chunks = []
    for step in instructions or []:
        if not isinstance(step, dict):
            continue
        title = step.get('step_title') or ''
        content = step.get('content') or ''
        if title:
            chunks.append(str(title))
        if content:
            chunks.append(str(content))
    return ' '.join(chunks)


def jaccard_similarity(left, right):
    if not left or not right:
        return 0.0
    union = left | right
    if not union:
        return 0.0
    return len(left & right) / len(union)


def combined_similarity(target, candidate):
    name_score = SequenceMatcher(None, target['name_norm'], candidate['name_norm']).ratio()
    ingredient_score = jaccard_similarity(target['ingredient_tokens'], candidate['ingredient_tokens'])
    instruction_score = jaccard_similarity(target['instruction_tokens'], candidate['instruction_tokens'])
    category_score = 1.0 if target['category_norm'] and target['category_norm'] == candidate['category_norm'] else 0.0

    if not target['ingredient_tokens']:
        ingredient_weight = 0.15
        instruction_weight = 0.15
    else:
        ingredient_weight = 0.3
        instruction_weight = 0.2

    return 0.4 * name_score + ingredient_weight * ingredient_score + instruction_weight * instruction_score + 0.1 * category_score


def weighted_median(pairs):
    ordered = sorted(pairs, key=lambda pair: pair[0])
    threshold = sum(weight for _, weight in ordered) / 2
    running = 0.0
    for value, weight in ordered:
        running += weight
        if running >= threshold:
            return int(round(value))
    return int(round(ordered[-1][0]))


def dispersion_ratio(values, center):
    if not values or center <= 0:
        return 1.0
    deviations = [abs(value - center) for value in values]
    mad = statistics.median(deviations)
    return mad / center


def candidate_terms(feature):
    terms = set(feature['name_tokens'])
    terms |= feature['ingredient_tokens']
    if len(feature['instruction_tokens']) <= 20:
        terms |= feature['instruction_tokens']
    return {term for term in terms if len(term) > 1}


def make_llm_payload(recipe):
    compact = {
        'dish_name': recipe.get('dish_name'),
        'category': recipe.get('category'),
        'ingredients': [
            {
                'name': item.get('name'),
                'quantity': item.get('quantity'),
                'unit': item.get('unit'),
            }
            for item in (recipe.get('ingredients') or [])
            if isinstance(item, dict)
        ],
        'instructions': [
            {
                'step_title': step.get('step_title'),
                'content': step.get('content'),
            }
            for step in (recipe.get('instructions') or [])
            if isinstance(step, dict)
        ],
        'prep_time_min': recipe.get('prep_time_min'),
    }
    return json.dumps(compact, ensure_ascii=False)


def estimate_with_llm(recipe, api_key, model, timeout_seconds):
    prompt = f"""You estimate recipe cooking time in minutes.
Return only valid JSON with key cook_time_min.
Rules:
- cook_time_min must be an integer >= 0.
- Estimate only cooking time, not prep time.
- If the recipe is a dip, sauce, or drink with no real heating, return a very low value.

Recipe:
{make_llm_payload(recipe)}"""

    response = requests.post(
        'https://api.openai.com/v1/responses',
        headers={
            'Authorization': f'Bearer {api_key}',
            'Content-Type': 'application/json',
        },
        json={
            'model': model,
            'input': prompt,
            'text': {
                'format': {
                    'type': 'json_schema',
                    'name': 'cook_time_estimate',
                    'schema': {
                        'type': 'object',
                        'properties': {
                            'cook_time_min': {'type': 'integer', 'minimum': 0},
                        },
                        'required': ['cook_time_min'],
                        'additionalProperties': False,
                    },
                }
            },
        },
        timeout=timeout_seconds,
    )
    response.raise_for_status()
    payload = response.json()

    text = None
    for item in payload.get('output', []):
        for content in item.get('content', []):
            if content.get('type') == 'output_text' and content.get('text'):
                text = content['text']
                break
        if text:
            break

    if not text:
        raise ValueError(f'Could not find text output in response: {payload}')

    parsed = json.loads(text)
    return int(parsed['cook_time_min'])



## Impute cook_time_min theo similar rồi mới dùng llm


In [12]:
records = df.to_dict(orient='records')

for recipe in records:
    recipe['cook_time_source'] = 'original' if int(recipe.get('cook_time_min') or 0) > 0 else ''

features = []
for idx, recipe in enumerate(records):
    feature = {
        'idx': idx,
        'dish_name': str(recipe.get('dish_name') or ''),
        'name_norm': normalize_text(str(recipe.get('dish_name') or '')),
        'name_tokens': tokenize(str(recipe.get('dish_name') or ''), keep_stopwords=True),
        'ingredient_tokens': tokenize(ingredient_text(recipe.get('ingredients') or [])),
        'instruction_tokens': tokenize(flatten_instruction_text(recipe.get('instructions') or [])),
        'category_norm': normalize_text(str(recipe.get('category') or '')),
        'cook_time_min': int(recipe.get('cook_time_min') or 0),
    }
    features.append(feature)

candidate_index = defaultdict(set)
non_missing_ids = set()
for feature in features:
    if feature['cook_time_min'] <= 0:
        continue

    non_missing_ids.add(feature['idx'])

    terms = candidate_terms(feature)
    for term in terms:
        candidate_index[term].add(feature['idx'])

    if feature['category_norm']:
        candidate_index[f"cat:{feature['category_norm']}"] .add(feature['idx'])


load_dotenv(DOTENV_PATH)
api_key = os.environ.get('OPENAI_API_KEY', '').strip() if USE_LLM_FALLBACK else None
llm_remaining = LLM_LIMIT
similar_count = 0
llm_count = 0

print("BƯỚC 5: IMPUTE COOK_TIME_MIN")
print(f"Số lượng recipe có cook_time_min sẵn: {sum(1 for r in records if r['cook_time_source'] == 'original')}")
print(f"Số lượng recipe cần xử lý thêm: {sum(1 for r in records if r['cook_time_source'] == '')}")

for idx, recipe in enumerate(records):
    if int(recipe.get('cook_time_min') or 0) > 0:
        continue

    target = features[idx]
    counts = Counter()
    for term in candidate_terms(target):
        for candidate_id in candidate_index.get(term, ()): 
            counts[candidate_id] += 1
    if target['category_norm']:
        for candidate_id in candidate_index.get(f"cat:{target['category_norm']}", ()): 
            counts[candidate_id] += 1

    if counts:
        candidate_ids = [candidate_id for candidate_id, _ in counts.most_common(300)]
    elif target['category_norm'] and candidate_index.get(f"cat:{target['category_norm']}"):
        candidate_ids = list(candidate_index.get(f"cat:{target['category_norm']}"))[:300]
    else:
        candidate_ids = list(non_missing_ids)[:300]

    scored = []
    for candidate_id in candidate_ids:
        candidate = features[candidate_id]
        if candidate['cook_time_min'] <= 0:
            continue
        score = combined_similarity(target, candidate)
        if score >= 0.35:
            scored.append((candidate, score))

    scored.sort(key=lambda pair: pair[1], reverse=True)
    similar_value = None
    if scored:
        top = scored[:10]
        top_score = top[0][1]
        selected = [pair for pair in top if pair[1] >= max(0.55, top_score - 0.12)][:5]

        if selected:
            estimate = weighted_median([(item['cook_time_min'], max(score, 0.01)) for item, score in selected])
            spread = dispersion_ratio([item['cook_time_min'] for item, _ in selected], estimate)
            same_name = any(item['name_norm'] == target['name_norm'] and item['name_norm'] for item, _ in selected)

            accept = (
                same_name
                or (top_score >= 0.82 and len(selected) >= 1 and spread <= 0.7)
                or (top_score >= 0.7 and len(selected) >= 2 and spread <= 0.45)
                or (top_score >= 0.62 and len(selected) >= 3 and spread <= 0.3)
            )

            if accept and estimate > 0:
                similar_value = int(estimate)

    if similar_value is not None:
        recipe['cook_time_min'] = similar_value
        recipe['cook_time_source'] = 'similar'
        similar_count += 1
        continue

    if USE_LLM_FALLBACK:
        if not api_key:
            raise RuntimeError('OPENAI_API_KEY was not found in environment or .env file.')
        if llm_remaining is None or llm_remaining > 0:
            llm_value = estimate_with_llm(recipe, api_key=api_key, model=LLM_MODEL, timeout_seconds=LLM_TIMEOUT_SECONDS)
            recipe['cook_time_min'] = int(llm_value)
            recipe['cook_time_source'] = 'llm'
            llm_count += 1
            if llm_remaining is not None:
                llm_remaining -= 1

print(f"\nKết quả impute:")
print(f"  - original: {sum(1 for r in records if r['cook_time_source'] == 'original')}")
print(f"  - similar: {sum(1 for r in records if r['cook_time_source'] == 'similar')}")
print(f"  - llm: {sum(1 for r in records if r['cook_time_source'] == 'llm')}")
print(f"  - còn thiếu source: {sum(1 for r in records if r['cook_time_source'] == '')}")


BƯỚC 5: IMPUTE COOK_TIME_MIN
Số lượng recipe có cook_time_min sẵn: 11545
Số lượng recipe cần xử lý thêm: 2079

Kết quả impute:
  - original: 11545
  - similar: 167
  - llm: 1912
  - còn thiếu source: 0


## Tổng kết xử lý và lưu kết quả

In [ ]:
processed = []
for recipe in records:
    final_recipe = dict(recipe)
    for col in ['prep_time', 'cook_time', 'servings', 'servings_numeric']:
        if col in final_recipe:
            del final_recipe[col]
    processed.append(final_recipe)

summary = {
    'total_rows': len(processed),
    'original_rows': sum(1 for r in processed if r.get('cook_time_source') == 'original'),
    'similar_rows': sum(1 for r in processed if r.get('cook_time_source') == 'similar'),
    'llm_rows': sum(1 for r in processed if r.get('cook_time_source') == 'llm'),
    'missing_rows': sum(1 for r in processed if not r.get('cook_time_source')),
}

print("TỔNG KẾT XỬ LÝ")
print(summary)

Path(OUTPUT_PATH).write_text(json.dumps(processed, ensure_ascii=False, indent=2), encoding='utf-8')
Path(OUTPUT_2_PATH).write_text(json.dumps(processed, ensure_ascii=False, indent=2), encoding='utf-8')
Path(SUMMARY_PATH).write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

print(f"Dữ liệu đã được lưu vào: {OUTPUT_PATH}")
print(f"Summary đã được lưu vào: {SUMMARY_PATH}")


TỔNG KẾT XỬ LÝ
{'total_rows': 13624, 'original_rows': 11545, 'similar_rows': 167, 'llm_rows': 1912, 'missing_rows': 0}
Dữ liệu đã được lưu vào: recipes_processed.json
Summary đã được lưu vào: recipes_processed_summary.json


## Xem nhanh dữ liệu đã xử lý xong


In [14]:
processed[:5]

[{'dish_name': 'Vịt quay tỳ bà',
  'url': 'https://www.dienmayxanh.com/vao-bep/cach-lam-vit-quay-ty-ba-da-gion-rum-mau-dep-thom-ngon-chuan-24176',
  'difficulty': 3,
  'views': 16,
  'ingredients': [{'name': 'Vịt', 'quantity': 1.0, 'unit': 'con'},
   {'name': 'Gừng', 'quantity': 100.0, 'unit': 'gr'},
   {'name': 'Rượu trắng', 'quantity': 50.0, 'unit': 'ml'},
   {'name': 'Mật ong', 'quantity': 3.0, 'unit': 'muỗng canh'},
   {'name': 'Mạch nha', 'quantity': 2.0, 'unit': 'muỗng canh'},
   {'name': 'Giấm đỏ', 'quantity': 2.0, 'unit': 'muỗng canh'},
   {'name': 'Bột tạo giòn', 'quantity': 1.0, 'unit': 'muỗng canh'},
   {'name': 'Lòng trắng trứng gà', 'quantity': 1.0, 'unit': 'quả'},
   {'name': 'Vừng trắng', 'quantity': 10.0, 'unit': 'gr'},
   {'name': 'Muối', 'quantity': 1.0, 'unit': 'ít'},
   {'name': 'Đường', 'quantity': 1.0, 'unit': 'ít'},
   {'name': 'Ngũ vị hương', 'quantity': 1.0, 'unit': 'ít'},
   {'name': 'Hành tím', 'quantity': 1.0, 'unit': 'ít'},
   {'name': 'Tỏi', 'quantity': 1.